# 43. 美化、注释与导出

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 12 / 12 步：组合、美化并交付完整报告**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 子图与组合图（subplots）  →  **本章任务：** 美化、注释与导出  →  **下一步：** 模块大作业《经营周会一页报告》
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

你辛苦算出来的结论，最终要交给一张图去讲给听众——无论是课堂汇报、实习报告还是作品集展示。



## 本章目标

学完本章，你将能够：

- **理解**：理解「美化、注释与导出」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「美化、注释与导出」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「美化、注释与导出」并读出其中的结论。


## 43.1 适用场景

**背景引入**：你辛苦算出来的结论，最终要交给一张图去讲给听众——无论是课堂汇报、实习报告还是作品集展示。可如果图上的标题、单位、注释和导出格式都没整理好，读者往往找不到重点，甚至可能误读结论。本章就来解决这个"最后一公里"问题：学会给图表做必要的修饰、加上让人一眼看懂的注释，并规范地导出图片，让结论清清楚楚。 打个比方：图画完只是草稿，美化注释就像给报告「写清标题、标好单位、配好图注」，再导出成高清 PNG/PDF，才像印刷品一样能交到别人手上——这一步是让结论清清楚楚抵达读者的「最后一公里」。

图表进入报告、汇报或作品集前的统一整理阶段。


## 43.2 数据结构

任何已经确定分析结论的Matplotlib图表。


## 43.3 本章练习任务

运行基础图表后，完成以下任务：

1. 修改 annotate 的 xytext 参数（如 (-40, 35) 或 (-70, 20)），调整注释箭头位置
2. 将 savefig 的 dpi 从 180 改为 300，对比不同分辨率的导出效果
3. 修改 grid 的 alpha 参数（如 0.05 或 0.3），说明网格透明度对可读性的影响


## 43.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`ax.bar()`、`ax.bar_label()`、`ax.yaxis.set_major_formatter()` | 图表进入报告、汇报或作品集前的统一整理阶段。 | 装饰多于信息 |
| 进阶变体 | `plt.subplots()`、`ax.plot()`、`profit.argmax()`、`ax.annotate()` | 在基础图表上增加分组、注释、布局或交互 | 颜色数量过多 |
| 关键参数 | `tick formatter` | 刻度格式 | 装饰多于信息 |
| 关键参数 | `annotate` | 注释 | 颜色数量过多 |
| 关键参数 | `spines` | 边框 | 数据标签互相遮挡 |
| 关键参数 | `savefig` | 导出 | 导出时标题被裁切 |


## 43.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = completed["InvoiceDate"].dt.to_period("M").astype("string")

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(rows["Country"], rows["flow"], values=rows["amount"].abs(), aggfunc="sum")
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 43.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
from matplotlib.ticker import FuncFormatter

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
colors = ["#9aa0a6"] * 5 + ["#1a73e8"]
bars = ax.bar(months, sales, color=colors)
ax.bar_label(bars, padding=4, fmt="%.0f")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0f}万"))
ax.set(title="6月销售额达到半年最高", xlabel="月份", ylabel="销售额")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
fig.tight_layout()
plt.show()


**练一练**：修改图表参数，观察高亮月份的切换

`38.4 基础图表` 的柱状图默认把 **6月**（最后一个柱子）用蓝色高亮，其余月份用灰色。请修改柱子的 `color` 列表，把高亮改到 **第一个月**（索引 0），并运行自检。

观察：高亮位置切换后，读者的视觉焦点从"结尾的最高柱"移到了"开头的起步柱"——同一份数据，因为一个参数变化，讲出的故事侧重点就不同了。用一两句话把这种变化记录下来。


In [ ]:
# 请在下方填写代码
from matplotlib.ticker import FuncFormatter

# 与本小节示例结构一致：6 个月份，销售额单位为万元
months = ["1月", "2月", "3月", "4月", "5月", "6月"]
sales = [52, 61, 58, 66, 70, 78]
# 练一练：默认高亮 6 月（最后一个柱子）。
# 请改成高亮第一个月（索引 0）。_X_ 处需填写让颜色总数与柱子数量一致的数值。
colors = ["#1a73e8"] + ["#9aa0a6"] * _X_


In [ ]:
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月", "4月", "5月", "6月"]
sales = [52, 61, 58, 66, 70, 78]

fig, ax = plt.subplots(figsize=(8, 4.2))
# 完整答案：_X_ = len(months) - 1，把高亮从末月（6月）改到首月（1月）
colors = ["#1a73e8"] + ["#9aa0a6"] * (len(months) - 1)
bars = ax.bar(months, sales, color=colors)
for _b in bars:  # 柱顶标注数值
    ax.text(
        _b.get_x() + _b.get_width() / 2,
        _b.get_height(),
        f"{_b.get_height():.0f}",
        ha="center",
        va="bottom",
    )
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0f}万"))
ax.set(title="改为高亮1月（完整答案）", xlabel="月份", ylabel="销售额")
for _sp in ["top", "right", "left"]:
    ax.spines[_sp].set_visible(False)
ax.grid(axis="y", alpha=0.15)


## 43.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, profit, marker="o", color="#188038", linewidth=2)
peak = int(profit.argmax())
ax.annotate(
    f"最高 {profit[peak]} 万元",
    (months[peak], profit[peak]),
    xytext=(-60, 28),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#188038"},
)
ax.set(title="利润在6月达到最高", ylabel="利润（万元）")
fig.tight_layout()
buffer = BytesIO()
fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
print(f"导出PNG大小: {buffer.getbuffer().nbytes / 1024:.1f} KB")
plt.show()


## 43.8 参数说明

- tick formatter：刻度格式
- annotate：注释
- spines：边框
- savefig：导出


## 43.9 结果解读

视觉重点应与结论一致；标题表达结论，坐标轴表达指标和单位。


## 43.10 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月", "4月"]
sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(months, sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 43.10.1 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 43.10.2 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 43.11 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月"]
sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(months, sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 43.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 43.12 易错点提醒

- 装饰多于信息
- 颜色数量过多
- 数据标签互相遮挡
- 导出时标题被裁切


## 43.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 43.14 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：突出最小值而非最大值，观察强调对象变化后的解读
# 【目标】同一组数据，换一个强调对象，读者会得出不同结论——练习这种设计选择。
import matplotlib.pyplot as plt

# 起点示例(已可运行)：用 np.argmin 找到最小月份，把它标红。
highlight = np.argmin(sales)
colors = ["#9aa0a6"] * len(months)
colors[highlight] = "#d93025"
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar(months, sales, color=colors)
ax.bar_label(bars, padding=4, fmt="%.0f")
ax.set(title="哪个月销售额最低？", xlabel="月份", ylabel="销售额")
ax.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout()
plt.show()

# ---- 反思记录：强调最大值 vs 最小值，结论导向有何不同 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
sales_growth = np.r_[np.nan, np.diff(sales) / sales[:-1]]
ax.plot(months, sales_growth * 100, marker="o", color="#1a73e8")
ax.axhline(0, color="#9aa0a6", linewidth=1)
ax.set(title="除3月外，月度销售额保持增长", ylabel="环比增长率（%）")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
fig.tight_layout()
plt.show()


## 43.15 小结

通过有限配色、刻度格式、重点注释和规范导出提升图表可读性。


### 43.15.1 你已经掌握

- 判断美化、注释与导出的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 43.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `tick formatter` | 刻度格式 |
| `annotate` | 注释 |
| `spines` | 边框 |
| `savefig` | 导出 |


### 43.15.3 需要注意

- 装饰多于信息
- 颜色数量过多
- 数据标签互相遮挡
- 导出时标题被裁切


### 43.15.4 完成检查

- [ ] 能判断什么问题适合使用美化、注释与导出
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 43.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
